# Attention-Improved Deep Learning Framework for Malaria and Tuberculosis
**Author:** Muhammed Toheeb Abdulraheem

This notebook orchestrates the training, evaluation, and interpretability of 5 deep learning architectures using custom CBAM attention layers.


In [ ]:
import sys
import os
import matplotlib.pyplot as plt

# Add src to path
sys.path.append(os.path.abspath('src'))

from data_loader import load_malaria_data, load_tb_data
from models import build_custom_cnn_attention, build_resnet50_attention, build_vgg16_attention, build_mobilenetv2_attention, build_densenet121_attention
from train import compile_model, train_model, unfreeze_and_finetune
from utils import plot_training_history, make_gradcam_heatmap, display_gradcam


## 1. Load Datasets
Ensure you have run `python src/download_data.py` to acquire the data.

In [ ]:
# Load Malaria Data
print("Loading Malaria Dataset...")
base_dir = os.path.abspath('.')
malaria_train, malaria_val = load_malaria_data(base_dir, batch_size=32)

# Load TB Data (Uncomment to use TB data instead)
# print("Loading TB Dataset...")
# tb_train, tb_val = load_tb_data(base_dir, batch_size=32)


## 2. Build Models with Attention
We can instantiate any of the 5 models here. Let's start with the lightweight MobileNetV2 for fast cloud execution.

In [ ]:
# Choose your architecture
# model = build_custom_cnn_attention()
# model = build_resnet50_attention()
# model = build_vgg16_attention()
model = build_mobilenetv2_attention()
# model = build_densenet121_attention()

# Compile
model = compile_model(model, learning_rate=1e-4)
model.summary()


## 3. Train Model
Training the top layers while the base convolutional layers are frozen.

In [ ]:
# Train the model
# Change malaria_train to tb_train if training for Tuberculosis
history = train_model(
    model, 
    train_data=malaria_train, 
    val_data=malaria_val, 
    epochs=15, 
    model_path='best_malaria_mobilenet_attention.h5'
)

plot_training_history(history, model_name=model.name)


## 4. Fine-Tuning
Unfreeze the top layers and fine-tune with a lower learning rate.

In [ ]:
# Fine-tune the top 20 layers
history_ft = unfreeze_and_finetune(
    model, 
    train_data=malaria_train, 
    val_data=malaria_val, 
    layers_to_unfreeze=20, 
    epochs=10, 
    learning_rate=1e-5
)

plot_training_history(history_ft, model_name=f"{model.name}_FineTuned")


## 5. Interpretability: Grad-CAM
Visualizing what the attention layers are focusing on.

In [ ]:
# Example Grad-CAM execution (Requires an image path)
import numpy as np
from utils import get_img_array, make_gradcam_heatmap, display_gradcam

# img_path = 'data/malaria/cell_images/cell_images/Parasitized/C100P61ThinF_IMG_20150918_144104_cell_162.png'
# img_array = get_img_array(img_path, size=(224, 224))

# For MobileNetV2, the last conv layer is often 'out_relu' or you can inspect model.summary()
# last_conv_layer_name = 'out_relu'

# heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name)
# display_gradcam(img_path, heatmap)
